# **KPI Engineering:**
**Objective:**
This step focuses on transforming baseline statistics into high-value metrics that reveal a deeper level of military intelligence. Through the calculation of specific KPIs, we aim to measure relative strategic capability—evaluating, for example, how a defense budget scales against a country's economic output rather than focusing on absolute figures. To support this, we will also enrich the data with regional and continental identifiers, ensuring that our final interactive dashboards have the grouping and filtering features required for comprehensive global comparisons.

In [5]:
import pandas as pd
import numpy as np

In [6]:
# Define input and output file names for the dataset
INPUT_FILE = "military_cleaned.csv"
OUTPUT_FILE = "military_final.csv"

# STEP 1: LOAD CLEAN DATASET
print(f"Step 1: Loading '{INPUT_FILE}'...")
try:
    # Attempt to load the cleaned military dataset from a CSV file
    df = pd.read_csv(INPUT_FILE)
    print(f"   -> Loaded {len(df)} rows.")
except FileNotFoundError:
    # If the input file is not found, raise an exception and guide the user to previous steps
    raise Exception(f"ERROR: Could not find '{INPUT_FILE}'. Please complete Task 6 first.")

# Clean column names: strip whitespace and convert to lowercase for consistent access
df.columns = df.columns.str.strip().str.lower()

# STEP 2: METADATA ENRICHMENT
print("\nStep 2: Adding Metadata (Continent, Region, Alliance, Flag Image URL)...")

# List of NATO member countries used to create an alliance flag
nato_list = [
    "albania", "belgium", "bulgaria", "canada", "croatia", "czechia",
    "denmark", "estonia", "finland", "france", "germany", "greece",
    "hungary", "iceland", "italy", "latvia", "lithuania", "luxembourg",
    "montenegro", "netherlands", "north-macedonia", "norway", "poland",
    "portugal", "romania", "slovakia", "slovenia", "spain", "sweden",
    "turkiye", "united-kingdom", "united-states-of-america", "united-states"
]

# Create 'alliance_flag' column: 'NATO' if country is in nato_list, 'Non-Aligned' otherwise
df['alliance_flag'] = (
    df['country']
    .str.lower()
    .str.strip()
    .str.replace(' ', '-', regex=False) # Standardize country names for matching with nato_list
    .apply(lambda x: 'NATO' if x in nato_list else 'Non-Aligned')
)

# Pre-compute a lookup dictionary for faster country-to-geo mapping
# This improves performance over iterating through the geo_mapping list for each country
geographical_lookup = {}
geographical_mapping = {
    'americas_northern': (['united-states', 'canada'], 'Americas', 'Northern America'),
    'americas_south': (['chile', 'brazil', 'argentina', 'colombia', 'peru', 'ecuador', 'venezuela', 'uruguay', 'paraguay', 'bolivia', 'suriname'], 'Americas', 'South America'),
    'americas_central': (['mexico', 'cuba', 'panama', 'guatemala', 'belize', 'dominican-republic', 'el-salvador', 'honduras', 'nicaragua'], 'Americas', 'Central America'),
    'asia_southern': (['india', 'pakistan', 'afghanistan', 'bangladesh', 'sri-lanka', 'nepal', 'bhutan'], 'Asia', 'Southern Asia'),
    'asia_eastern': (['china', 'japan', 'korea', 'taiwan', 'mongolia'], 'Asia', 'Eastern Asia'),
    'asia_southeast': (['vietnam', 'thailand', 'indonesia', 'philippines', 'singapore', 'malaysia', 'cambodia', 'laos', 'myanmar'], 'Asia', 'South-East Asia'),
    'asia_central': (['kazakhstan', 'uzbekistan', 'turkmenistan', 'kyrgyzstan', 'tajikistan'], 'Asia', 'Central Asia'),
    'europe_eastern': (['russia', 'ukraine', 'poland', 'romania', 'belarus', 'czechia', 'hungary', 'bulgaria', 'moldova', 'albania'], 'Europe', 'Eastern Europe'),
    'europe_western': (['france', 'germany', 'belgium', 'netherlands', 'switzerland', 'austria', 'luxembourg'], 'Europe', 'Western Europe'),
    'europe_northern': (['united-kingdom', 'sweden', 'norway', 'denmark', 'finland', 'ireland', 'iceland', 'estonia', 'latvia', 'lithuania'], 'Europe', 'Northern Europe'),
    'europe_southern': (['italy', 'spain', 'greece', 'portugal', 'serbia', 'croatia', 'slovenia', 'bosnia', 'montenegro', 'north-macedonia', 'kosovo'], 'Europe', 'Southern Europe'),
    'middle_east': (['israel', 'saudi', 'turkiye', 'iran', 'iraq', 'uae', 'qatar', 'jordan', 'kuwait', 'lebanon', 'oman', 'yemen', 'bahrain', 'syria', 'cyprus', 'georgia', 'armenia', 'azerbaijan'], 'Middle East', 'Western Asia (Middle East)'),
    'africa_northern': (['egypt', 'algeria', 'morocco', 'libya', 'tunisia'], 'Africa', 'Northern Africa'),
    'africa_subsaharan': (['nigeria', 'south-africa', 'kenya', 'ethiopia', 'angola', 'benin', 'botswana', 'burundi', 'cameroon', 'central-african-republic', 'chad', 'congo', 'djibouti', 'eritrea', 'gabon', 'gambia', 'ghana', 'guinea', 'ivory-coast', 'lesotho', 'liberia', 'madagascar', 'malawi', 'mali', 'mauritania', 'mauritius', 'mozambique', 'namibia', 'niger', 'senegal', 'sierra-leone', 'somalia', 'sudan', 'tanzania', 'uganda', 'zambia', 'zimbabwe'], 'Africa', 'Sub-Saharan Africa'),
    'oceania': (['australia', 'new-zealand'], 'Oceania', 'Australia & New Zealand')
}

# Populate the lookup dictionary
for _, (country_list, continent, region) in geographical_mapping.items():
    for country_name_fragment in country_list:
        geographical_lookup[country_name_fragment] = (continent, region)

# Function to map countries to continents and regions using a pre-computed lookup
def get_geo_details_optimized(country_name):
    # Standardize country name for lookup
    c = str(country_name).lower().strip().replace(' ', '-')

    # Attempt direct lookup
    for key_fragment, (continent, region) in geographical_lookup.items():
        if key_fragment in c:
            return continent, region

    # If no match is found, assign 'Other' for continent and subregion
    return 'Other', 'Other Subregion'

# Apply the optimized geographic details function to create 'continent' and 'region' columns
df[['continent', 'region']] = df['country'].apply(lambda x: pd.Series(get_geo_details_optimized(x)))

# Function to generate a plausible flag image URL (e.g., from Wikipedia or similar pattern)
def get_flag_image_url(country_name):
    # Standardize country name for URL (e.g., 'United-States-of-America' -> 'United_States_of_America')
    normalized_country_name = str(country_name).replace(' ', '_').replace('-', '_')
    # Example URL pattern (adjust if a specific source is preferred)
    return f"https://upload.wikimedia.org/wikipedia/commons/thumb/b/b3/Flag_of_{normalized_country_name}.svg/25px-Flag_of_{normalized_country_name}.svg.png"

# Apply the flag image URL function to create the 'flag_image_url' column
df['flag_image_url'] = df['country'].apply(get_flag_image_url)

print("   -> 'continent', 'region', 'alliance_flag', and 'flag_image_url' columns successfully mapped/generated.")

# STEP 3: KPI CALCULATIONS
# Define a function to encapsulate all KPI calculation logic
def calculate_kpis(df: pd.DataFrame) -> pd.DataFrame:
    print("\nStep 3: Calculating KPIs (encapsulated in function)...")

    # Define numeric columns that are used in KPI calculations
    num_cols = ['total_aircraft', 'tanks', 'naval_assets', 'total_manpower', 'defense_budget', 'gdp']
    # Convert these columns to numeric types, coercing errors to NaN and filling NaN with 0
    for col in num_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    # Safely extract 'power_index', convert to numeric, and handle missing values.
    # Missing values are filled with 999 to place them at the bottom of rankings.
    if 'power_index' in df.columns:
        # Extract numeric part (e.g., '1.23'), convert to float, and fill NaNs with 999
        df['power_index'] = pd.to_numeric(df['power_index'].astype(str).str.extract(r'([\d.]+)')[0], errors='coerce').fillna(999)
    else:
        # Assign a default high value (999) if 'power_index' column is missing
        df['power_index'] = 999

    # KPI 1: Calculate total military hardware by summing relevant asset columns (aircraft, tanks, naval assets)
    df['total_military_hardware'] = df.get('total_aircraft', 0) + df.get('tanks', 0) + df.get('naval_assets', 0)

    # KPI 2: Calculate Assets per Capita. Handle potential division by zero by replacing 0 manpower with 1.
    df['assets_per_capita'] = (df['total_military_hardware'] / df.get('total_manpower', 1).replace(0, 1)).round(7)

    # KPI 3: Calculate Budget-to-GDP Ratio. Handle potential division by zero by replacing 0 GDP with 1.
    df['budget_gdp_ratio'] = (df.get('defense_budget', 0) / df.get('gdp', 1).replace(0, 1)).round(5)

    # KPI 4: Calculate Power Index Rank and determine gap from the top rank
    df['rank'] = df['power_index'].rank(ascending=True, method='min').astype(int) # Rank countries by power_index (lower is better)
    top_rank_value = df['rank'].min() # Get the best (lowest) rank value
    df['power_index_rank_gap'] = df['rank'] - top_rank_value # Calculate how far each country is from the top rank

    print("   -> KPIs successfully calculated.")
    return df

# Call the KPI calculation function
df = calculate_kpis(df)

# STEP 5: VALIDATE KPI VALUES
print("\nStep 5: Validating Data...")

# Check for and handle infinite values in 'budget_gdp_ratio' (can occur from division by zero before fillna)
if np.isinf(df['budget_gdp_ratio']).any():
    print("   WARNING: Infinite values found in Budget-to-GDP. Replacing with 0.")
    df.replace([np.inf, -np.inf], 0, inplace=True) # Replace infinities with 0

# Display a sample for validation (e.g., India's metrics) to confirm calculations
sample = df[df['country'] == 'India'][['country', 'assets_per_capita', 'budget_gdp_ratio', 'rank']]
if not sample.empty:
    print(f"   -> Validation Sample (India):\n{sample.to_string(index=False)}")
else:
    print("   -> Validation Sample (India) not found.") # Inform if India is not in the dataset

# STEP 6: EXPORT FINAL DATASET
print(f"\nStep 6: Exporting to '{OUTPUT_FILE}'...")
# Export the final processed DataFrame to a new CSV file without the pandas index
df.to_csv(OUTPUT_FILE, index=False)

print(f"[SUCCESS] Final analytics file ready: {OUTPUT_FILE}")

Step 1: Loading 'military_cleaned.csv'...
   -> Loaded 145 rows.

Step 2: Adding Metadata (Continent, Region, Alliance, Flag Image URL)...
   -> 'continent', 'region', 'alliance_flag', and 'flag_image_url' columns successfully mapped/generated.

Step 3: Calculating KPIs (encapsulated in function)...
   -> KPIs successfully calculated.

Step 5: Validating Data...
   -> Validation Sample (India):
country  assets_per_capita  budget_gdp_ratio  rank
  India            0.00001           0.00765     4

Step 6: Exporting to 'military_final.csv'...
[SUCCESS] Final analytics file ready: military_final.csv


**Takeaway:** Appended 3 geographic dimensions and successfully calculated 3 analytical KPIs (`assets_per_capita`, `budget_gdp_ratio`, `power_index_rank_gap`) across all 145 rows, exporting to `military_final.csv`.